# 02 — Light Curve Gallery

Visualize example light curves for every class. Build physical intuition about
the morphological differences between variable stars and transients.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG, VIS_CONFIG
from src.data import get_object_lightcurve, get_objects_by_class
from src.utils import plot_lightcurve, get_class_name, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
lc = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet'))
print(f"Loaded {len(metadata)} objects, {len(lc):,} observations")

## 1. Light Curve Gallery — 3 Examples per Class

For each of the 14 classes, plot 3 randomly selected light curves to see the diversity
of morphologies within each class.

In [ ]:
n_per_class = 3
classes = sorted(DATA_CONFIG['class_names'].keys())
n_classes = len(classes)

fig, axes = plt.subplots(n_classes, n_per_class, figsize=(5 * n_per_class, 3.5 * n_classes))

for row, target in enumerate(classes):
    obj_ids = get_objects_by_class(metadata, target, n=n_per_class)
    class_name = get_class_name(target)
    for col, obj_id in enumerate(obj_ids):
        ax = axes[row, col]
        object_lc = get_object_lightcurve(obj_id, lc)
        plot_lightcurve(object_lc, object_id=obj_id,
                        title=f"{class_name} ({target}) — {obj_id}",
                        ax=ax, show_errors=False)
        if col > 0:
            ax.set_ylabel('')
        ax.legend(fontsize=6, ncol=3)

plt.tight_layout()
save_figure(fig, 'lightcurve_gallery')
plt.show()

## 2. Class Morphology Notes

Observations about each class (fill in after viewing the gallery):

- **SN Ia (90):** Classic rise-then-decay, red bands peak later than blue
- **SN II (42):** Longer plateau phase after peak, slower decline
- **SN Ibc (62):** Similar to Ia but typically fainter, faster decline
- **SN Iax (52):** Subluminous Ia-like, lower amplitude
- **SN Ia-91bg (67):** Fast-declining Ia subclass
- **SLSN-I (95):** Very bright, slow evolution over months
- **Kilonova (64):** Extremely fast transient, fades in days
- **TDE (15):** Smooth rise and power-law decline over months
- **AGN (88):** Stochastic variability, no clear period, long duration
- **RR Lyrae (92):** Periodic, short period (~0.2-0.8 days), sawtooth shape
- **Mira (53):** Periodic, long period (~100-500 days), large amplitude
- **Eclipsing Binary (16):** Periodic dips, period from hours to days
- **M-dwarf Flare (65):** Sharp spike then exponential decay
- **Microlensing (6):** Achromatic symmetric bump (same shape in all bands)

## 3. Observation Cadence

In [ ]:
# Number of observations per object per band
obs_per_obj_band = lc.groupby(['object_id', 'passband']).size().reset_index(name='n_obs')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total observations per object
obs_per_obj = lc.groupby('object_id').size()
axes[0].hist(obs_per_obj, bins=80, edgecolor='black', linewidth=0.3)
axes[0].set_xlabel('Total Observations per Object')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Total Observations (median={obs_per_obj.median():.0f})')

# Per-band observation count
for band_id, band_name in DATA_CONFIG['bands'].items():
    band_obs = obs_per_obj_band[obs_per_obj_band['passband'] == band_id]['n_obs']
    axes[1].hist(band_obs, bins=40, alpha=0.5,
                 label=f"{band_name} (med={band_obs.median():.0f})",
                 color=VIS_CONFIG['band_colors'][band_id])

axes[1].set_xlabel('Observations per Band per Object')
axes[1].set_ylabel('Count')
axes[1].set_title('Per-Band Observation Counts')
axes[1].legend(fontsize=8)

plt.tight_layout()
save_figure(fig, 'cadence_distribution')
plt.show()

In [ ]:
# Time between consecutive observations (cadence)
lc_sorted = lc.sort_values(['object_id', 'mjd'])
dt = lc_sorted.groupby('object_id')['mjd'].diff().dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(dt[dt < 50], bins=200, edgecolor='black', linewidth=0.2)
ax.set_xlabel('Time Between Observations (days)')
ax.set_ylabel('Count')
ax.set_title(f'Inter-Observation Cadence (median={dt.median():.1f} days)')
plt.tight_layout()
plt.show()